# Step 3 — Missing Value Detection & Treatment
### Credit Card Underwriting Pipeline

---

## Why Missing Values Are a Problem

> **Every machine learning algorithm (except a few tree-based methods) requires complete data.**
> A single `NaN` in a row will cause most models to either crash or silently produce wrong predictions.

Real-world data is almost never complete. Reasons for missing values:
- Applicant skipped a form field
- System error during data collection
- Feature was not collected in older applications (structural missingness)
- Sensitive fields that applicants refused to answer

---

## The Three Types of Missingness

Understanding *why* data is missing determines *how* to handle it:

| Type | Meaning | Example | Strategy |
|------|---------|---------|----------|
| **MCAR** | Missing Completely At Random | Random system glitch | Impute or drop rows |
| **MAR** | Missing At Random (depends on other observed columns) | High earners skip income field | Impute using related columns |
| **MNAR** | Missing Not At Random (depends on the value itself) | People with bad credit skip credit score | Hardest — imputing is risky |

> In credit modelling, **missingness itself is a signal** — a missing fraud_risk_score often
> indicates a thin-file applicant (little credit history), which is itself a risk indicator.

---

## Our Strategy (from the Solution Notebook)

1. **Audit** — count missing values per column and compute percentage
2. **Visualise** — bar chart to spot severely missing columns
3. **Drop** — remove columns with > 50% missing (too little data to impute reliably)
4. **Impute numeric** → median (robust to outliers and skew)
5. **Impute categorical** → mode (most frequent value)


In [ ]:
# ── Re-run Setup ──────────────────────────────────────────────────────────────
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')

DATA_PATH   = 'cc_underwriting_5k_stratified11.csv'
IGNORE_COLS = ['applicant_id', 'target_approved', 'target_credit_limit_assigned']
SEED = 42

df_raw       = pd.read_csv(DATA_PATH)
numeric_cols = [c for c in df_raw.select_dtypes(include='number').columns if c not in IGNORE_COLS]
cat_cols     = [c for c in df_raw.select_dtypes(include=['object','string']).columns if c not in IGNORE_COLS]

print(f'Loaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')

## 3.1 — Missing Value Audit

The `missing_report` function calculates:
- **`missing_count`** — raw number of nulls in the column
- **`missing_pct`** — percentage of rows that are null

> **Rule of thumb in credit modelling:**
> - < 5% missing → safe to impute
> - 5–50% missing → impute carefully, consider adding a "was_missing" flag
> - > 50% missing → usually drop the column (not enough signal)


In [ ]:
def missing_report(df):
    """
    Create a summary table of missing values.

    Parameters:
        df : DataFrame to audit

    Returns:
        DataFrame sorted by missing percentage (descending)
        Only shows columns that have at least 1 missing value
    """
    missing      = df.isnull().sum()           # count nulls per column
    pct          = (missing / len(df)) * 100   # convert to percentage
    report       = pd.DataFrame({
        'missing_count': missing,
        'missing_pct'  : pct.round(2)
    })
    # Filter to only columns WITH missing values, sorted worst-first
    report = report[report['missing_count'] > 0].sort_values('missing_pct', ascending=False)
    return report

# Run the audit on the raw dataset
miss = missing_report(df_raw)

print(f'Total columns    : {df_raw.shape[1]}')
print(f'Columns with NaN : {len(miss)}')
print(f'Columns complete : {df_raw.shape[1] - len(miss)}')
print()

if len(miss) > 0:
    print('Columns with missing values (sorted by % missing):')
    print(miss.to_string())
else:
    print('No missing values found in this dataset.')

## 3.2 — Visualise Missingness

> A horizontal bar chart makes it instantly obvious which columns are most affected.
> The dashed line at 50% is our decision threshold — columns to the right get dropped.


In [ ]:
# Only plot if there are missing columns
if len(miss) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(miss) * 0.35)))

    # Horizontal bar chart: feature name on y-axis, % missing on x-axis
    miss['missing_pct'].plot(kind='barh', ax=ax,
                              color='#e74c3c', edgecolor='white', alpha=0.85)

    # Draw the 50% decision threshold line
    ax.axvline(x=50, color='black', linestyle='--', linewidth=1.5,
               label='50% threshold (drop above this)')

    ax.set_xlabel('Missing %', fontsize=12)
    ax.set_title('Missing Values per Column', fontsize=14, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No chart needed — no missing values.')

## 3.3 — Drop High-Missingness Columns

> **Why drop rather than impute columns with > 50% missing?**
>
> If more than half of a column's values are missing, any imputed value we put there
> is mostly made-up data. The column will contain more noise than signal.
> The model might learn from the imputed values and underperform on new data.
>
> **Exception:** If domain knowledge tells you the missingness itself is meaningful
> (e.g., `fraud_score` being missing = thin file), you can keep it by adding a binary
> `was_missing` indicator column instead.


In [ ]:
HIGH_MISS_THRESH = 50.0   # drop columns with more than 50% missing

# Get list of columns to drop
drop_cols = miss[miss['missing_pct'] > HIGH_MISS_THRESH].index.tolist()

print(f'Columns to drop ({len(drop_cols)} total):')
for c in drop_cols:
    pct_val = miss.loc[c, 'missing_pct']
    print(f'  {c}: {pct_val:.1f}% missing')

# Drop them from a copy of the raw DataFrame
# We call this df (working DataFrame) from now on
df = df_raw.drop(columns=drop_cols)

# Update our feature lists to remove any dropped columns
numeric_cols = [c for c in numeric_cols if c in df.columns]
cat_cols     = [c for c in cat_cols     if c in df.columns]

print()
print(f'Shape before drop : {df_raw.shape}')
print(f'Shape after drop  : {df.shape}')
print(f'Columns removed   : {df_raw.shape[1] - df.shape[1]}')

## 3.4 — Impute Remaining Missing Values

### Why Median for Numeric?

> Mean is pulled towards outliers. For right-skewed data like income or net worth,
> the mean can be much higher than where most values sit.
> **Median is always the middle value** — it does not move when outliers exist.
>
> Example: incomes of [30k, 35k, 40k, 45k, 1,000k]
> Mean = 230k (distorted by the millionaire)
> Median = 40k (accurate centre of the group)

### Why Mode for Categorical?

> We cannot compute an average of "Full-Time" and "Part-Time."
> **Mode = most common value** — a safe, unbiased fill for categorical missingness.
>
> Alternative: treat missing as its own category "Unknown" — useful when missing is informative.


In [ ]:
# Track imputation actions for documentation / audit trail
imputation_log = []

# ── Numeric Columns: Fill with Median ────────────────────────────────────────
print('NUMERIC IMPUTATION (median):')
for col in numeric_cols:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)

        imputation_log.append({
            'column': col, 'type': 'numeric',
            'strategy': 'median', 'fill_value': median_val, 'n_filled': n_miss
        })
        print(f'  {col:<45} filled {n_miss:>4} NaN with median={median_val:.3f}')

# ── Categorical Columns: Fill with Mode ──────────────────────────────────────
print()
print('CATEGORICAL IMPUTATION (mode):')
for col in cat_cols:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        mode_val = df[col].mode()[0]   # mode() returns a Series; [0] gets the top value
        df[col].fillna(mode_val, inplace=True)

        imputation_log.append({
            'column': col, 'type': 'categorical',
            'strategy': 'mode', 'fill_value': mode_val, 'n_filled': n_miss
        })
        print(f'  {col:<45} filled {n_miss:>4} NaN with mode="{mode_val}"')

# ── Verify No Remaining Nulls ─────────────────────────────────────────────────
remaining_nulls = df.isnull().sum().sum()
print()
print(f'Total remaining nulls: {remaining_nulls}')
print(f'Imputation complete: {len(imputation_log)} columns were filled')

## 3.5 — (Advanced) Alternative: Add Missing Indicator Features

> **When missingness itself carries information, do not just impute — signal it.**
>
> For example: if `fraud_risk_score` is missing, this often means the bureau could not
> generate a score (thin file), which is itself a risk signal.
>
> By adding a binary `fraud_risk_score_was_missing` column, the model can learn from
> both the imputed value AND the fact that it was originally missing.
>
> This approach is used in production credit scorecards at major banks.


In [ ]:
# Add "was_missing" binary indicator columns BEFORE imputing
# (In a real pipeline, run this before the imputation step above)

# Columns where missingness is informative (domain knowledge call)
INFORMATIVE_MISS = ['fraud_risk_score', 'predicted_default_probability',
                    'avg_bureau_score', 'synthetic_identity_score']

# Filter to columns that actually had missing values
informative_cols_present = [c for c in INFORMATIVE_MISS if c in df_raw.columns]

df_indicators = df.copy()  # work on a separate copy for demonstration
for col in informative_cols_present:
    n_miss_orig = df_raw[col].isnull().sum()
    if n_miss_orig > 0:
        # Create a binary column: 1 = was missing, 0 = was present
        df_indicators[col + '_was_missing'] = df_raw[col].isnull().astype(int)
        print(f'  Added {col}_was_missing  ({n_miss_orig} missing originally)')

print()
print('NOTE: This notebook uses the simpler median/mode imputation approach.')
print('In a production scorecard, combine both: impute + add indicator columns.')

## Step 3 Complete — Missing Value Summary

**What we did:**

| Action | Details |
|--------|---------|
| Audited all 200 columns | Created `missing_report()` with count + % |
| Visualised missingness | Bar chart with 50% threshold line |
| Dropped high-miss columns | Removed columns with > 50% missing |
| Imputed numeric (median) | Robust to skew and outliers |
| Imputed categorical (mode) | Most frequent category fill |
| Discussed indicators | "Was_missing" flag for informative missingness |

**Outcome:** `df` now has zero missing values and is ready for cleaning and feature engineering.

**Next:** `04_Data_Cleaning.ipynb` — Outlier detection, type corrections, encoding strategies
